<a href="https://colab.research.google.com/github/VitorEduardoOliveira/NLP/blob/main/ChatBot_Array.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio sentence-transformers numpy

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ── Modelo multilíngue (suporta português nativamente) ─────────────────────────
modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# ── Base de conhecimento ───────────────────────────────────────────────────────
conhecimento = [
    ("qual é a capital do brasil",                 "A capital do Brasil é Brasília, inaugurada em 21 de abril de 1960."),
    ("qual é a capital de são paulo",              "A capital do estado de São Paulo é a cidade de São Paulo."),
    ("quem foi santos dumont",                     "Santos Dumont foi um inventor brasileiro pioneiro da aviação, nascido em 1873 em Minas Gerais."),
    ("o que é machine learning",                   "Machine Learning é uma área da IA onde modelos aprendem padrões automaticamente a partir de dados."),
    ("o que é deep learning",                      "Deep Learning usa redes neurais profundas para aprender representações complexas dos dados."),
    ("o que é python",                             "Python é uma linguagem de programação de alto nível, famosa pela simplicidade e versatilidade."),
    ("como funciona a similaridade de cossenos",   "A similaridade de cossenos mede o ângulo entre dois vetores: quanto menor o ângulo, mais similares são os textos."),
    ("o que é nlp",                                "NLP é o campo da IA que ensina máquinas a entender e gerar linguagem humana."),
    ("o que é tfidf",                              "TF-IDF pondera palavras pela frequência no documento e pela raridade no corpus inteiro."),
    ("o que é sentence transformers",              "Sentence Transformers convertem frases inteiras em vetores densos, capturando semântica real."),
    ("o que é gradio",                             "Gradio cria interfaces web interativas para modelos de ML com poucas linhas de código."),
    ("como usar google colab",                     "O Google Colab é um ambiente de notebooks Python gratuito que roda na nuvem."),
    ("o que é uma rede neural",                    "Uma rede neural é um modelo computacional inspirado no cérebro humano, com camadas de neurônios artificiais."),
    ("o que é inteligência artificial",            "IA é o campo da computação dedicado a criar sistemas que simulam capacidades humanas."),
    ("o que é um vetor de embeddings",             "Embeddings são representações numéricas de texto onde frases similares ficam próximas no espaço vetorial."),
    ("olá",                                        "Olá! Sou um chatbot semântico com Sentence Transformers. Pergunte-me algo!"),
    ("oi",                                         "Oi! Como posso ajudar você hoje?"),
    ("obrigado",                                   "De nada! Fico feliz em ajudar."),
    ("tchau",                                      "Até mais! Foi um prazer conversar com você."),
]

perguntas = [p for p, _ in conhecimento]
respostas = [r for _, r in conhecimento]

# ── Pré-computar embeddings da base de conhecimento ───────────────────────────
print("Gerando embeddings da base de conhecimento...")
embeddings_base = modelo.encode(perguntas, convert_to_numpy=True)
print("Pronto! " + str(len(perguntas)) + " entradas indexadas.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Gerando embeddings da base de conhecimento...
Pronto! 19 entradas indexadas.


In [ ]:
def responder(pergunta_usuario: str, limiar: float = 0.40) -> str:
    if not pergunta_usuario.strip():
        return "Por favor, digite uma pergunta."

    embedding_usuario = modelo.encode([pergunta_usuario], convert_to_numpy=True)
    similaridades     = cosine_similarity(embedding_usuario, embeddings_base).flatten()

    melhor_idx   = int(np.argmax(similaridades))
    melhor_score = float(similaridades[melhor_idx])

    if melhor_score >= limiar:
        return f"{respostas[melhor_idx]}\n\n📊 similaridade: {melhor_score:.2f}"
    else:
        return f"❓ Não encontrei resposta suficientemente similar (score: {melhor_score:.2f}). Tente reformular!"

# Teste rápido antes de subir a interface
print(responder("me fala sobre inteligência artificial"))
print(responder("o que fazem as sentence transformers?"))

IA é o campo da computação dedicado a criar sistemas que simulam capacidades humanas.

📊 similaridade: 0.93
Sentence Transformers convertem frases inteiras em vetores densos, capturando semântica real.

📊 similaridade: 0.96


In [ ]:
import gradio as gr

# ── Exemplos clicáveis ─────────────────────────────────────────────────────────
exemplos = [
    ["o que é machine learning?"],
    ["como funciona a similaridade de cossenos?"],
    ["quem foi santos dumont?"],
    ["o que são embeddings?"],
    ["o que é gradio?"],
]

# ── Layout da interface ────────────────────────────────────────────────────────
with gr.Blocks(title="Chatbot Semântico", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🤖 Chatbot
    """)

    chatbot = gr.Chatbot(
        label="Conversa",
        height=420,
        bubble_full_width=False,
    )

    with gr.Row():
        campo = gr.Textbox(
            placeholder="Digite sua pergunta e pressione Enter...",
            label="",
            scale=5,
            autofocus=True,
        )
        btn = gr.Button("Enviar", variant="primary", scale=1)

    limiar_slider = gr.Slider(
        minimum=0.1, maximum=0.95, value=0.40, step=0.05,
        label="Limiar de similaridade",
        info="Abaixo desse valor o bot diz que não sabe responder"
    )

    gr.Examples(exemplos, inputs=campo, label="Exemplos de perguntas")

    # ── Lógica de turno ───────────────────────────────────────────────────────
    def turno(mensagem, historico, limiar):
        if not mensagem.strip():
            return historico, ""
        resposta = responder(mensagem, limiar)
        historico = historico + [(mensagem, resposta)]
        return historico, ""

    btn.click(
        fn=turno,
        inputs=[campo, chatbot, limiar_slider],
        outputs=[chatbot, campo],
    )
    campo.submit(
        fn=turno,
        inputs=[campo, chatbot, limiar_slider],
        outputs=[chatbot, campo],
    )

# ── Subir com link público (share=True gera URL ngrok temporária) ──────────────
demo.launch(share=True)

/tmp/ipykernel_1125/3479504100.py:13: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Chatbot Semântico", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_1125/3479504100.py:19: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_1125/3479504100.py:19: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipykernel_1125/3479504100.py:19: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to 

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://39db6885e7e64a6700.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
